# Restoration suitability raster preparation for the Mujib Basin Digital Twin

This notebook processes the Marab-barley and Vallerani-saltbush suitability rasters
from Haddad et al. (2024) for use in the dashboard. It corresponds to Section 3.3.5
of the thesis.

The published rasters combine seven environmental criteria (rainfall deficit, temperature,
slope, topographic wetness index, land use, soil texture, and soil pH) through a
multi-criteria evaluation weighted by the Analytic Hierarchy Process.

**Processing steps:**

1. Clip the national-scale suitability GeoTIFFs to the Mujib Basin boundary
2. Clean nodata and zero values within the clipped extent
3. Apply the 80% suitability threshold to generate candidate-status binary layers
4. Reproject from UTM Zone 37N to WGS84 on a 1655 x 2000 pixel grid
5. Produce grayscale value rasters and colour-rendered PNGs for the dashboard
6. Generate metadata JSON with geographic bounds and class breaks

**Inputs:** Haddad et al. (2024) suitability GeoTIFFs (UTM Zone 37N, approx. 1 km resolution)

**Outputs:** Dashboard-ready PNGs and metadata in `runtime-data/suitability/`

**Thesis reference:** Section 3.3.5 (methodology), Section 4.5 (results), Figure 3.3


## 1. Configuration and imports


In [ ]:
import json
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.warp import reproject, transform_bounds, Resampling
from rasterio.transform import from_bounds
from pathlib import Path
from PIL import Image
import geopandas as gpd

# Repository paths
REPO_ROOT = Path("..")
CESIUM_DIR = REPO_ROOT / "runtime-data"
SUIT_DIR = REPO_ROOT / "suitability-source"  # Source GeoTIFFs (not in repository; obtain from ICARDA)
SUIT_OUT = REPO_ROOT / "runtime-data" / "suitability"
BASIN_GJ = REPO_ROOT / "runtime-data" / "boundaries" / "basin_OFFICIAL.geojson"

SUIT_OUT.mkdir(parents=True, exist_ok=True)

# Source raster filenames
MARAB_TIF = SUIT_DIR / "Marab_barley_suitability_values_mujib_cleaned.tif"
VALLERANI_TIF = SUIT_DIR / "Vallerani_sultbush_suitability_values_mujib_cleaned.tif"

print("Suitability source dir:", SUIT_DIR)
print("Output dir:", SUIT_OUT)


## 2. Clip suitability rasters to the Mujib Basin

The published rasters cover all of Jordan. This step clips them to the Mujib Basin
boundary and masks nodata and zero-value pixels. The basin boundary is reprojected
to match the raster CRS (UTM Zone 37N, EPSG:32637) before clipping.


In [ ]:

# CLIP SUITABILITY VALUES TIFFS TO MUJIB BASIN
# AND MASK HIDDEN NODATA + ZERO VALUES


import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# INPUT PATHS
# ------------------------------------------------------------
basin_shp = str(REPO_ROOT / "placeholder")  # Update path

value_tifs = [
    str(SUIT_DIR / "Marab_barley_suitability_values.tif"),
    str(SUIT_DIR / "Vallerani_sultbush_suitability_values.tif")
]

# Hidden ArcGIS fill value
HIDDEN_NODATA = -3.402823e+38

# Output nodata to write properly
OUTPUT_NODATA = -9999.0

# ------------------------------------------------------------
# READ BASIN SHAPEFILE
# ------------------------------------------------------------
gdf = gpd.read_file(basin_shp)

if gdf.empty:
    raise ValueError("Mujib basin shapefile is empty.")

print("Basin shapefile CRS:", gdf.crs)

# ------------------------------------------------------------
# LOOP THROUGH TIFFS
# ------------------------------------------------------------
for tif_path in value_tifs:
    print("\n" + "=" * 90)
    print("Processing:", tif_path)

    with rasterio.open(tif_path) as src:
        print("Raster CRS:", src.crs)
        print("Raster shape:", (src.height, src.width))
        print("Raster resolution:", src.res)

        # Reproject basin to raster CRS if needed
        if gdf.crs != src.crs:
            gdf_raster = gdf.to_crs(src.crs)
            print("Reprojected basin shapefile to raster CRS.")
        else:
            gdf_raster = gdf.copy()

        # Convert geometry to GeoJSON-like mappings
        geoms = [geom for geom in gdf_raster.geometry if geom is not None]

        if len(geoms) == 0:
            raise ValueError("No valid geometries found in Mujib basin shapefile.")

        # Clip raster to basin
        out_image, out_transform = mask(
            src,
            geoms,
            crop=True,
            filled=True,
            nodata=OUTPUT_NODATA
        )

        # Since these are single-band rasters
        arr = out_image[0].astype("float32")

        # Mask hidden nodata and zero values
        invalid_mask = (
            np.isclose(arr, HIDDEN_NODATA) |
            np.isclose(arr, 0.0) |
            np.isclose(arr, OUTPUT_NODATA)
        )

        arr[invalid_mask] = OUTPUT_NODATA

        # Update metadata/profile
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": arr.shape[0],
            "width": arr.shape[1],
            "transform": out_transform,
            "count": 1,
            "dtype": "float32",
            "nodata": OUTPUT_NODATA,
            "compress": "lzw"
        })

        # Output file name
        out_path = str(Path(tif_path).with_name(Path(tif_path).stem + "_mujib_cleaned.tif"))

        # Save clipped raster
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(arr, 1)

        # ----------------------------------------------------
        # Quick stats
        # ----------------------------------------------------
        valid = arr[arr != OUTPUT_NODATA]

        print("Saved:", out_path)
        print("Clipped shape:", arr.shape)
        print("Valid pixel count:", valid.size)

        if valid.size > 0:
            print("Min valid:", float(valid.min()))
            print("Max valid:", float(valid.max()))
            print("Mean valid:", float(valid.mean()))
            print("Std valid:", float(valid.std()))
        else:
            print("No valid pixels inside Mujib basin.")

        # ----------------------------------------------------
        # Preview plot
        # ----------------------------------------------------
        plt.figure(figsize=(7, 6))
        masked_plot = np.where(arr == OUTPUT_NODATA, np.nan, arr)
        plt.imshow(masked_plot, cmap="viridis")
        plt.colorbar(label="Suitability (%)")
        plt.title(Path(out_path).name)
        plt.xlabel("Column")
        plt.ylabel("Row")
        plt.tight_layout()
        plt.show()

## 3. Apply suitability threshold and compute statistics

The 80% threshold from Haddad et al. (2024) is applied to generate candidate-status
binary layers. Basin-level statistics are computed for benchmarking against the
published Mujib values (Section 4.5).


In [ ]:
# ============================================================
# COUNT VALID VS WHITE/MASKED AREA IN MUJIB CLIPPED RASTERS
# ============================================================


import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------
basin_shp = str(REPO_ROOT / "placeholder")  # Update path

clipped_tifs = [
    str(SUIT_DIR / "Marab_barley_suitability_values_mujib_cleaned.tif"),
    str(SUIT_DIR / "Vallerani_sultbush_suitability_values_mujib_cleaned.tif")
]

OUTPUT_NODATA = -9999.0

# ------------------------------------------------------------
# READ BASIN
# ------------------------------------------------------------
gdf = gpd.read_file(basin_shp)

if gdf.empty:
    raise ValueError("Basin shapefile is empty.")

results = []

for tif_path in clipped_tifs:
    with rasterio.open(tif_path) as src:
        arr = src.read(1)

        # Reproject basin to raster CRS if needed
        if gdf.crs != src.crs:
            gdf_r = gdf.to_crs(src.crs)
        else:
            gdf_r = gdf.copy()

        geoms = [geom for geom in gdf_r.geometry if geom is not None]

        # basin mask on THIS clipped raster grid
        # True = inside basin
        basin_mask = geometry_mask(
            geoms,
            transform=src.transform,
            invert=True,
            out_shape=(src.height, src.width)
        )

        # valid and nodata
        valid_mask = (arr != OUTPUT_NODATA)
        nodata_mask = (arr == OUTPUT_NODATA)

        # counts
        inside_basin_pixels = np.sum(basin_mask)
        valid_inside_pixels = np.sum(basin_mask & valid_mask)
        white_inside_pixels = np.sum(basin_mask & nodata_mask)
        outside_basin_pixels = np.sum(~basin_mask)

        # pixel area in km²
        pixel_area_km2 = abs(src.res[0] * src.res[1]) / 1_000_000.0

        # areas
        inside_basin_area_km2 = inside_basin_pixels * pixel_area_km2
        valid_inside_area_km2 = valid_inside_pixels * pixel_area_km2
        white_inside_area_km2 = white_inside_pixels * pixel_area_km2
        outside_basin_area_km2 = outside_basin_pixels * pixel_area_km2

        # percentages within basin only
        valid_pct_in_basin = (valid_inside_pixels / inside_basin_pixels * 100) if inside_basin_pixels > 0 else np.nan
        white_pct_in_basin = (white_inside_pixels / inside_basin_pixels * 100) if inside_basin_pixels > 0 else np.nan

        results.append({
            "file": Path(tif_path).name,
            "pixel_area_km2": pixel_area_km2,
            "inside_basin_pixels": int(inside_basin_pixels),
            "valid_inside_pixels": int(valid_inside_pixels),
            "white_inside_pixels": int(white_inside_pixels),
            "outside_basin_pixels": int(outside_basin_pixels),
            "inside_basin_area_km2": inside_basin_area_km2,
            "valid_inside_area_km2": valid_inside_area_km2,
            "white_inside_area_km2": white_inside_area_km2,
            "outside_basin_area_km2": outside_basin_area_km2,
            "valid_pct_in_basin": valid_pct_in_basin,
            "white_pct_in_basin": white_pct_in_basin
        })

# ------------------------------------------------------------
# SHOW RESULTS
# ------------------------------------------------------------
df = pd.DataFrame(results)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\nSUMMARY TABLE\n")
print(df)

# Nice readable print
for _, row in df.iterrows():
    print("\n" + "=" * 90)
    print(row["file"])
    print("=" * 90)
    print(f"Pixel area                : {row['pixel_area_km2']:.3f} km²")
    print(f"Inside basin area         : {row['inside_basin_area_km2']:.2f} km²")
    print(f"Valid colored area        : {row['valid_inside_area_km2']:.2f} km²")
    print(f"White/masked area         : {row['white_inside_area_km2']:.2f} km²")
    print(f"Outside-basin bbox area   : {row['outside_basin_area_km2']:.2f} km²")
    print(f"Valid % inside basin      : {row['valid_pct_in_basin']:.2f}%")
    print(f"White % inside basin      : {row['white_pct_in_basin']:.2f}%")

## 4. Reproject and render dashboard PNGs

The clipped rasters are reprojected to WGS84 and resampled to the 1655 x 2000 pixel
grid shared with the NDVI overlays. Continuous suitability surfaces and candidate-zone
overlays are exported as PNGs with a metadata JSON storing bounds and class breaks.


In [ ]:

# SUITABILITY -> CESIUM-READY PNG + JSON
# Mujib Basin | Marab + Vallerani
# ============================================================


import json
from pathlib import Path

import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import reproject, transform_bounds, Resampling
from PIL import Image

# ------------------------------------------------------------
# INPUT CLEANED TIFFS
# ------------------------------------------------------------
SRC_FILES = {
    "marab": str(SUIT_DIR / "Marab_barley_suitability_values_mujib_cleaned.tif"),
    "vallerani": str(SUIT_DIR / "Vallerani_sultbush_suitability_values_mujib_cleaned.tif"),
}

# Output folder for Cesium-ready assets
OUT_DIR = SUIT_OUT
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# OUTPUT GRID
# Keep same dashboard-style size family as NDVI workflow.
# You can change these later if needed.
# ------------------------------------------------------------
OUT_WIDTH = 1655
OUT_HEIGHT = 2000
OUT_CRS = "EPSG:4326"

# ------------------------------------------------------------
# VALID RANGES + CLASS COLORS
# Using the study/derived ranges you already confirmed.
# ------------------------------------------------------------
LAYER_CONFIG = {
    "marab": {
        "title": "Marab suitability",
        "system": "Marab + barley",
        "vmin": 40.342185974121094,
        "vmax": 94.93281555175781,
        "classes": [
            {"min": 40.34, "max": 50.00, "label": "40.34 - 50", "color": "#F44316"},
            {"min": 50.00, "max": 60.00, "label": "50 - 60",    "color": "#F98A00"},
            {"min": 60.00, "max": 70.00, "label": "60 - 70",    "color": "#F2CD00"},
            {"min": 70.00, "max": 80.00, "label": "70 - 80",    "color": "#B7D300"},
            {"min": 80.00, "max": 90.00, "label": "80 - 90",    "color": "#6EAA00"},
            {"min": 90.00, "max": 94.93, "label": "90 - 94.93", "color": "#005A0A"},
        ],
    },
    "vallerani": {
        "title": "Vallerani suitability",
        "system": "Vallerani + saltbush",
        "vmin": 47.08399963378906,
        "vmax": 98.28604888916016,
        "classes": [
            {"min": 47.08, "max": 50.00, "label": "47.08 - 50", "color": "#F44316"},
            {"min": 50.00, "max": 60.00, "label": "50 - 60",    "color": "#F98A00"},
            {"min": 60.00, "max": 70.00, "label": "60 - 70",    "color": "#F2CD00"},
            {"min": 70.00, "max": 80.00, "label": "70 - 80",    "color": "#B7D300"},
            {"min": 80.00, "max": 90.00, "label": "80 - 90",    "color": "#6EAA00"},
            {"min": 90.00, "max": 98.29, "label": "90 - 98.29", "color": "#005A0A"},
        ],
    },
}

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def hex_to_rgb(hex_color: str):
    h = hex_color.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def encode_grayscale_rgba(arr: np.ndarray, vmin: float, vmax: float) -> np.ndarray:
    """
    Encode valid values to grayscale 0..255 with alpha=255.
    Invalid/NaN -> alpha=0.
    """
    rgba = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)

    valid = np.isfinite(arr)
    if np.any(valid):
        scaled = (arr[valid] - vmin) / (vmax - vmin)
        scaled = np.clip(scaled, 0, 1)
        gray = np.round(scaled * 255).astype(np.uint8)

        rgba[..., 0][valid] = gray
        rgba[..., 1][valid] = gray
        rgba[..., 2][valid] = gray
        rgba[..., 3][valid] = 255

    return rgba

def encode_classed_color_rgba(arr: np.ndarray, class_defs: list) -> np.ndarray:
    """
    Encode valid values into discrete class colors with transparent NoData.
    """
    rgba = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)

    valid = np.isfinite(arr)
    rgba[..., 3][valid] = 0  # set later for matched classes

    for i, cls in enumerate(class_defs):
        lo = cls["min"]
        hi = cls["max"]
        color = hex_to_rgb(cls["color"])

        if i < len(class_defs) - 1:
            mask = np.isfinite(arr) & (arr >= lo) & (arr < hi)
        else:
            mask = np.isfinite(arr) & (arr >= lo) & (arr <= hi)

        rgba[..., 0][mask] = color[0]
        rgba[..., 1][mask] = color[1]
        rgba[..., 2][mask] = color[2]
        rgba[..., 3][mask] = 255

    return rgba

# ------------------------------------------------------------
# STEP 1: COMPUTE COMMON WGS84 BOUNDS
# Use a shared rectangle for both layers so they align exactly.
# ------------------------------------------------------------
west_all, south_all = 999, 999
east_all, north_all = -999, -999

for key, src_path in SRC_FILES.items():
    with rasterio.open(src_path) as src:
        w, s, e, n = transform_bounds(src.crs, OUT_CRS, *src.bounds, densify_pts=21)
        west_all = min(west_all, w)
        south_all = min(south_all, s)
        east_all = max(east_all, e)
        north_all = max(north_all, n)

dst_transform = from_bounds(west_all, south_all, east_all, north_all, OUT_WIDTH, OUT_HEIGHT)

print("Common 4326 bounds:")
print("west =", west_all)
print("south =", south_all)
print("east =", east_all)
print("north =", north_all)
print("size =", OUT_WIDTH, "x", OUT_HEIGHT)

# ------------------------------------------------------------
# STEP 2: REPROJECT EACH TIFF TO COMMON 4326 GRID
# ------------------------------------------------------------
meta_out = {
    "crs": OUT_CRS,
    "west": west_all,
    "south": south_all,
    "east": east_all,
    "north": north_all,
    "width": OUT_WIDTH,
    "height": OUT_HEIGHT,
    "layers": {}
}

for key, src_path in SRC_FILES.items():
    cfg = LAYER_CONFIG[key]

    with rasterio.open(src_path) as src:
        src_arr = src.read(1).astype(np.float32)
        src_nodata = src.nodata

        # Turn nodata into NaN before reprojection
        if src_nodata is not None:
            src_arr = np.where(src_arr == src_nodata, np.nan, src_arr)

        dst_arr = np.full((OUT_HEIGHT, OUT_WIDTH), np.nan, dtype=np.float32)

        reproject(
            source=src_arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=OUT_CRS,
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.nearest  # suitability is classed -> keep crisp
        )

        # Optional safety clip to expected valid range
        dst_arr = np.where(np.isfinite(dst_arr), np.clip(dst_arr, cfg["vmin"], cfg["vmax"]), np.nan)

        # ----------------------------------------------------
        # SAVE GRAYSCALE VALUE PNG (optional but useful later)
        # ----------------------------------------------------
        value_rgba = encode_grayscale_rgba(dst_arr, cfg["vmin"], cfg["vmax"])
        value_png_name = f"{key}_suitability_value_4326.png"
        value_png_path = OUT_DIR / value_png_name
        Image.fromarray(value_rgba, mode="RGBA").save(value_png_path)

        # ----------------------------------------------------
        # SAVE COLOUR PNG OVERLAY
        # ----------------------------------------------------
        color_rgba = encode_classed_color_rgba(dst_arr, cfg["classes"])
        color_png_name = f"{key}_suitability_color_4326.png"
        color_png_path = OUT_DIR / color_png_name
        Image.fromarray(color_rgba, mode="RGBA").save(color_png_path)

        # ----------------------------------------------------
        # STORE JSON METADATA
        # ----------------------------------------------------
        valid = np.isfinite(dst_arr)
        meta_out["layers"][key] = {
            "title": cfg["title"],
            "system": cfg["system"],
            "value_png": value_png_name,
            "color_png": color_png_name,
            "vmin": cfg["vmin"],
            "vmax": cfg["vmax"],
            "valid_pixels": int(valid.sum()),
            "classes": cfg["classes"],
        }

        print(f"\n[{key}]")
        print("  saved:", value_png_path)
        print("  saved:", color_png_path)
        print("  valid pixels:", int(valid.sum()))
        if np.any(valid):
            print("  min:", float(np.nanmin(dst_arr)))
            print("  max:", float(np.nanmax(dst_arr)))
            print("  mean:", float(np.nanmean(dst_arr)))

# ------------------------------------------------------------
# STEP 3: WRITE JSON
# ------------------------------------------------------------
json_path = OUT_DIR / "suitability_bounds_scales.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(meta_out, f, indent=2)

print("\nSaved JSON:", json_path)
print("Done.")

## Summary

This notebook produced the suitability layers used by the dashboard:

- `marab_suitability_color_4326.png`: continuous Marab-barley suitability surface
- `vallerani_suitability_color_4326.png`: continuous Vallerani-saltbush suitability surface
- `marab_candidate_gt80_4326.png`: binary Marab candidate zones (above 80%)
- `vallerani_candidate_gt80_4326.png`: binary Vallerani candidate zones (above 80%)
- `combined_candidate_gt80_4326.png`: areas suitable for both interventions
- `candidate_overlay_bounds_all_thresholds.json`: bounds and metadata

Basin-level suitability statistics (Section 4.5):
- Marab-barley mean: 75.8%, area above 80%: 1,258 km2
- Vallerani-saltbush mean: 80.6%, area above 80%: 3,265 km2
- Published Mujib means (Haddad et al., 2024): 75.8% and 79.0%
- Deviation from published values: less than 0.1 pp (Marab) and 1.6 pp (Vallerani)

Thesis references: Section 3.3.5 (methodology), Section 4.5 (results), Figure 3.3.
